# Financial Market Statistical Analysis

This notebook provides a comprehensive statistical analysis of financial features using the refactored `finance_ml.analytics` module. 
It leverages advanced statistical methods (Bayesian, MCMC, Kalman Filters), interactive dashboards, and performance-optimized operations.


## 1. Setup and Environment Configuration
We import the core analytics modules and configure the visualization environment.


In [1]:
import logging
import warnings

import pandas as pd
import plotly.express as px

# Core analytics imports (consolidated)
from finance_ml.analytics import (
    # Data utilities
    backfill_feature_columns,
    # Statistical analysis
    bayesian_category_analysis,
    fit_distributions_by_category,
    hierarchical_mcmc_by_sector,
    kalman_momentum_filter,
    fit_gaussian_copula,
    # Optimized operations
    fast_ruin_probability,
    get_optimization_status,
    # Screening
    create_enhanced_screener,
    # Feature analytics & dashboards
    PLOTLY_TEMPLATE,
    create_interactive_momentum_dashboard,
    create_interactive_valuation_heatmap,
    create_leverage_liquidity_quadrant,
    bayesian_earnings_beat_model,
    analyze_distress_distribution,
    create_summary_dashboard,
)

# Profitability visualizations
from finance_ml.analytics.visualizations.profitability import (
    create_margin_waterfall_chart,
    create_dupont_decomposition_dashboard,
    create_profitability_quadrant,
)

# Technical visualizations
from finance_ml.analytics.visualizations.technical import (
    create_momentum_ribbon_chart,
    create_52w_range_distribution,
)

# Temporal analysis visualizations
from finance_ml.analytics.visualizations.temporal_analysis import (
    create_earnings_calendar_heatmap,
    create_inventory_cycle_analysis,
    create_fcf_trajectory_chart,
    create_dividend_streak_timeline,
)

# Category-specific chart functions
from finance_ml.analytics.visualizations.category_charts import (
    # Analyst Sentiment
    create_analyst_sentiment_histogram,
    create_analyst_upside_scatter,
    # Earnings Quality
    create_eps_surprise_histogram,
    create_eps_trajectory_scatter,
    # Growth Metrics
    create_growth_correlation_heatmap,
    create_revenue_vs_eps_growth_scatter,
    # Cash Flow
    create_fcf_margin_yield_scatter,
    create_cash_flow_quality_boxplot,
    # Dividend Features
    create_dividend_yield_payout_scatter,
    create_shareholder_yield_histogram,
    # R&D Investment
    create_rnd_intensity_boxplot,
    create_rnd_intensity_growth_scatter,
    create_rnd_per_employee_histogram,
    # Inventory
    create_inventory_days_turnover_scatter,
    # Goodwill & M&A
    create_goodwill_concentration_boxplot,
    create_goodwill_impairment_scatter,
    create_acquisition_activity_histogram,
    # CapEx & Investment
    create_capex_growth_scatter,
    create_investment_efficiency_boxplot,
    create_ma_intensity_histogram,
)

# Configuration
logging.basicConfig(level=logging.INFO)
warnings.filterwarnings("ignore")
px.defaults.template = PLOTLY_TEMPLATE


## 2. Data Acquisition
Loading 14 feature categories from the `mv_all_stock_features` materialized view.


In [2]:
# Feature Category Definitions
FEATURE_CATEGORIES = {
    'Valuation Ratios': ['p_e_ratio', 'p_b_ratio', 'ev_ebitda_ratio', 'ev_sales_ratio', 'dividend_yield', 'peg_ratio', 'price_to_tangible_book', 'tangible_book_value_ltm'],
    'Momentum & Technical': ['price_momentum_1m', 'price_momentum_3m', 'price_momentum_6m', 'price_momentum_1y', 'price_momentum_3y', 'price_momentum_5y', 'range_52w_position', 'long_term_trend_score', 'secular_trend_flag'],
    'Profitability': ['roe', 'roa', 'gross_margin_pct', 'operating_margin_pct', 'net_margin_pct', 'ebitda_margin_pct', 'roic', 'net_margin_trend_yoy'],
    'Quality & Risk': ['piotroski_f_score', 'distress_risk_score', 'altman_z_score', 'accounting_quality_score', 'earnings_quality_composite', 'cash_flow_quality_score', 'beta_stability_score'],
    'Leverage & Liquidity': ['debt_to_equity', 'current_ratio', 'quick_ratio', 'interest_coverage_ratio', 'cash_ratio', 'working_capital_ratio', 'debt_deleveraging'],
    'Analyst Sentiment': ['analyst_bullish_pct', 'analyst_neutral_pct', 'analyst_bearish_pct', 'upside_potential', 'analyst_rating_normalized', 'eps_revision_momentum'],
    'Earnings Quality': ['eps_surprise_pct', 'eps_adjustment_ratio', 'gaap_adj_eps_gap_pct', 'eps_trajectory_score', 'earnings_quality_score', 'gaap_revision_momentum'],
    'Growth Metrics': ['revenue_growth_yoy', 'ebitda_growth_yoy', 'eps_yoy_growth', 'fcf_growth_yoy', 'revenue_cagr_5y'],
    'Cash Flow': ['fcf_positive_years', 'fcf_margin', 'fcf_yield', 'cfo_to_net_income', 'self_funding_ratio', 'cash_flow_quality_score'],
    'Dividend Features': ['dividend_streak', 'dividend_yield_ltm', 'dividend_payout_ratio', 'fcf_dividend_coverage', 'total_shareholder_yield'],
    'R&D Investment': ['rnd_intensity_ltm', 'rnd_yoy_growth', 'rnd_per_employee', 'high_rnd_intensity_flag'],
    'Inventory Temporal': ['inventory_days', 'inventory_turnover', 'inventory_yoy_change', 'inventory_buildup_flag'],
    'Goodwill & M&A': ['goodwill_concentration', 'goodwill_3y_growth', 'recent_acquisition_flag', 'impairment_risk_score'],
    'CapEx & Investment': ['capex_yoy_growth', 'capex_vs_5y_avg', 'acquisitions_ltm_total', 'ma_intensity_score', 'investment_efficiency'],
}


In [3]:
%%sql
SELECT *
FROM public.mv_all_stock_features
WHERE next_earnings >= DATE '2026-01-01' and region= 'Europe'
ORDER BY next_earnings ASC;


,isin,ticker,name,industry,sector,trading_country,region,country,exchange,last_updated,...,other_unusual_items_ltm,impairment_goodwill_ltm,unusual_asset_writedown_ltm,restructuring_charges_ltm,total_unusual_items,unusual_items_to_revenue,unusual_items_to_ebitda,has_unusual_items_flag,earnings_quality_impact,feature_calculated_at
0,NL0015001KT6,BRE,Brembo N.V.,Automobile Components,Consumer Discretionary,IT,Europe,IT,BIT,2026-01-29,...,-5.15,0.00,-5.15,0.0,-10.30,0.234249,1.571823,1,96.155999,2026-01-30 10:05:43.792185+00
1,SE0009806607,MTRS,Munters Group AB (publ),Building Products,Industrials,SE,Europe,SE,OM,2026-01-29,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-01-30 10:05:43.792185+00
2,IE00BK9ZQ967,TT,Trane Technologies plc,Building Products,Industrials,US,Europe,IE,NYSE,2026-01-29,...,61.20,0.00,0.00,-31.2,30.00,0.140700,0.694300,1,98.972110,2026-01-30 10:05:43.792185+00
3,CH0012221716,ABBN,ABB Ltd,Electrical Equipment,Industrials,CH,Europe,CH,SWX,2026-01-29,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-01-30 10:05:43.792185+00
4,CH0210483332,CFR,Compagnie Financière Richemont SA,Textiles Apparel and Luxury Goods,Consumer Discretionary,CH,Europe,CH,SWX,2026-01-29,...,-3.52,-23.48,-3.52,0.0,-30.52,0.118486,0.486106,1,99.366855,2026-01-30 10:05:43.792185+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1896,CH0127480363,AUTN,Autoneum Holding AG,Automobile Components,Consumer Discretionary,CH,Europe,CH,SWX,2026-01-29,...,-5.04,0.00,1.13,0.0,-3.91,0.134995,1.807842,1,94.233889,2026-01-30 10:05:43.792185+00
1897,CH0006372897,INRN,Interroll Holding AG,Machinery,Industrials,CH,Europe,CH,SWX,2026-01-29,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-01-30 10:05:43.792185+00
1898,CH0010702154,KOMN,Komax Holding AG,Machinery,Industrials,CH,Europe,CH,SWX,2026-01-29,...,0.40,0.00,0.00,0.0,0.40,0.054487,1.287416,1,96.927803,2026-01-30 10:05:43.792185+00
1899,FR0004027068,ALLAN,Lanson-BCC,Beverages,Consumer Staples,FR,Europe,FR,ENXTPA,2026-01-29,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-01-30 10:05:43.792185+00


In [4]:
# Normalize SQL result and backfill expected columns using the utility function
if not isinstance(df, pd.DataFrame):
    try:
        df = df.DataFrame()
    except Exception:
        pass

if isinstance(df, pd.DataFrame):
    df = backfill_feature_columns(df)


INFO:root:Backfill complete. Columns: 716


## 3. Comprehensive Statistical Analysis by Category

This section provides in-depth statistical analysis across all 14 feature categories using Bayesian methods, distribution fitting, and specialized visualizations.


### 3.1 Valuation Ratios
We use Bayesian analysis to estimate true valuation means and visualize valuation metrics across industries.


In [5]:
val_results = bayesian_category_analysis(df, 'Valuation Ratios', FEATURE_CATEGORIES['Valuation Ratios'])
val_distributions = fit_distributions_by_category(df, 'Valuation Ratios', FEATURE_CATEGORIES['Valuation Ratios'])
create_interactive_valuation_heatmap(df).show()


### 3.2 Momentum & Technical
Applying Kalman filters to smooth momentum signals and visualizing multi-period momentum patterns.


In [6]:
df_kalman = kalman_momentum_filter(df, momentum_cols=['price_momentum_1y', 'price_momentum_3m'])
momentum_results = bayesian_category_analysis(df, 'Momentum & Technical', FEATURE_CATEGORIES['Momentum & Technical'])
create_interactive_momentum_dashboard(df).show()

In [7]:
create_52w_range_distribution(df).show()

In [8]:
create_momentum_ribbon_chart(df).show()


### 3.3 Profitability
Utilizing DuPont decomposition and margin waterfall charts to analyze bottom-line drivers with Bayesian estimation.


In [9]:
prof_results = bayesian_category_analysis(df, 'Profitability', FEATURE_CATEGORIES['Profitability'])
prof_distributions = fit_distributions_by_category(df, 'Profitability', FEATURE_CATEGORIES['Profitability'])
create_dupont_decomposition_dashboard(df).show()

In [10]:
create_margin_waterfall_chart(df).show()

In [11]:
create_profitability_quadrant(df).show()


### 3.4 Quality & Risk
Assessing quality scores and distress risk using Bayesian analysis and tail risk metrics.


In [12]:
quality_results = bayesian_category_analysis(df, 'Quality & Risk', FEATURE_CATEGORIES['Quality & Risk'])
quality_distributions = fit_distributions_by_category(df, 'Quality & Risk', FEATURE_CATEGORIES['Quality & Risk'])
analyze_distress_distribution(df).show()
df_ruin = fast_ruin_probability(df)


### 3.5 Leverage & Liquidity
Analyzing solvency metrics and balance sheet strength using quadrant analysis and Bayesian estimation.


In [40]:
leverage_results = bayesian_category_analysis(df, 'Leverage & Liquidity', FEATURE_CATEGORIES['Leverage & Liquidity'])
leverage_distributions = fit_distributions_by_category(df, 'Leverage & Liquidity', FEATURE_CATEGORIES['Leverage & Liquidity'])
create_leverage_liquidity_quadrant(df).show()


### 3.6 Analyst Sentiment
Analyzing analyst recommendations, price target upside, and EPS revision momentum.


In [41]:
sentiment_results = bayesian_category_analysis(df, 'Analyst Sentiment', FEATURE_CATEGORIES['Analyst Sentiment'])
sentiment_distributions = fit_distributions_by_category(df, 'Analyst Sentiment', FEATURE_CATEGORIES['Analyst Sentiment'])
create_analyst_sentiment_histogram(df).show()

In [15]:
create_analyst_upside_scatter(df).show()


### 3.7 Earnings Quality
Evaluating earnings surprises, GAAP adjustments, and earnings trajectory with Bayesian methods.


In [43]:
earnings_quality_results = bayesian_category_analysis(df, 'Earnings Quality', FEATURE_CATEGORIES['Earnings Quality'])
earnings_quality_distributions = fit_distributions_by_category(df, 'Earnings Quality', FEATURE_CATEGORIES['Earnings Quality'])
earnings_beat_probs = bayesian_earnings_beat_model(df)
create_eps_surprise_histogram(df).show()

In [42]:
create_eps_trajectory_scatter(df).show()


### 3.8 Growth Metrics
Analyzing revenue, EBITDA, EPS, and FCF growth patterns with distribution fitting.


In [18]:
growth_results = bayesian_category_analysis(df, 'Growth Metrics', FEATURE_CATEGORIES['Growth Metrics'])
growth_distributions = fit_distributions_by_category(df, 'Growth Metrics', FEATURE_CATEGORIES['Growth Metrics'])
create_growth_correlation_heatmap(df, FEATURE_CATEGORIES['Growth Metrics']).show()

In [19]:
create_revenue_vs_eps_growth_scatter(df).show()


### 3.9 Cash Flow
Analyzing free cash flow metrics, self-funding ratios, and cash flow quality.


In [20]:
cashflow_results = bayesian_category_analysis(df, 'Cash Flow', FEATURE_CATEGORIES['Cash Flow'])
cashflow_distributions = fit_distributions_by_category(df, 'Cash Flow', FEATURE_CATEGORIES['Cash Flow'])
create_fcf_trajectory_chart(df).show()

In [45]:
create_fcf_margin_yield_scatter(df).show()

In [46]:
create_cash_flow_quality_boxplot(df).show()


### 3.10 Dividend Features
Evaluating dividend sustainability, payout ratios, and shareholder yield.


In [23]:
dividend_results = bayesian_category_analysis(df, 'Dividend Features', FEATURE_CATEGORIES['Dividend Features'])
dividend_distributions = fit_distributions_by_category(df, 'Dividend Features', FEATURE_CATEGORIES['Dividend Features'])
create_dividend_streak_timeline(df).show()

In [47]:
create_dividend_yield_payout_scatter(df).show()

In [25]:
create_shareholder_yield_histogram(df).show()


### 3.11 R&D Investment
Analyzing R&D intensity, growth patterns, and innovation investment efficiency.


In [26]:
rnd_results = bayesian_category_analysis(df, 'R&D Investment', FEATURE_CATEGORIES['R&D Investment'])
rnd_distributions = fit_distributions_by_category(df, 'R&D Investment', FEATURE_CATEGORIES['R&D Investment'])
create_rnd_intensity_boxplot(df).show()

In [27]:
create_rnd_intensity_growth_scatter(df).show()

In [28]:
create_rnd_per_employee_histogram(df).show()


### 3.12 Inventory Temporal
Analyzing inventory cycles, turnover efficiency, and buildup patterns.


In [29]:
inventory_results = bayesian_category_analysis(df, 'Inventory Temporal', FEATURE_CATEGORIES['Inventory Temporal'])
inventory_distributions = fit_distributions_by_category(df, 'Inventory Temporal', FEATURE_CATEGORIES['Inventory Temporal'])
create_inventory_cycle_analysis(df).show()

In [48]:
create_inventory_days_turnover_scatter(df).show()


### 3.13 Goodwill & M&A
Evaluating acquisition activity, goodwill concentration, and impairment risk.


In [31]:
goodwill_results = bayesian_category_analysis(df, 'Goodwill & M&A', FEATURE_CATEGORIES['Goodwill & M&A'])
goodwill_distributions = fit_distributions_by_category(df, 'Goodwill & M&A', FEATURE_CATEGORIES['Goodwill & M&A'])
create_goodwill_concentration_boxplot(df).show()

In [32]:
create_goodwill_impairment_scatter(df).show()

In [33]:
create_acquisition_activity_histogram(df).show()


### 3.14 CapEx & Investment
Analyzing capital expenditure patterns, investment efficiency, and M&A intensity.


In [34]:
capex_results = bayesian_category_analysis(df, 'CapEx & Investment', FEATURE_CATEGORIES['CapEx & Investment'])
capex_distributions = fit_distributions_by_category(df, 'CapEx & Investment', FEATURE_CATEGORIES['CapEx & Investment'])
create_capex_growth_scatter(df).show()

In [35]:
create_investment_efficiency_boxplot(df).show()

In [36]:
create_ma_intensity_histogram(df).show()


## 4. Advanced Modeling: Hierarchical Bayes & Copulas
Modeling sector-level dependencies and tail correlations between Valuation and Quality.


In [37]:
roe_hierarchical = hierarchical_mcmc_by_sector(df, 'roe', sector_col='industry')
copula_fit = fit_gaussian_copula(df, ['p_e_ratio', 'piotroski_f_score'])


## 5. Earnings Calendar Analysis
Visualizing upcoming earnings dates with quality overlay.


In [38]:
create_earnings_calendar_heatmap(df).show()


## 6. Stock Screening & Summary
Final ranking and interactive dashboard for the top opportunities.


In [39]:
top_picks = create_enhanced_screener(df, min_fscore=7)
create_summary_dashboard(df).show()

# Optimization Summary
opt_status = get_optimization_status()
print(f"JIT Acceleration: {opt_status.get('numba_available')}")


JIT Acceleration: False
